# YOLO26s Fine-Tune — Thermal Human Detection from UAV (Night)

**Goal:** fine-tune YOLO26s pada dataset Roboflow `thermal-disasters-project/thermal-human-detection-from-uav` v1 (654 frame thermal UAV, kelas tunggal `Human`, kurasi untuk skenario malam).

**Setup dulu (sekali):**
1. Settings → Accelerator = **GPU T4x2**
2. API key Roboflow dibaca dari Kaggle Secret `ROBOFLOW_API_KEY` — jangan menaruh key di notebook.
3. Alternatif tanpa key: upload `model/data/night-vision/thermal-human-detection-from-uav.zip` sebagai Kaggle Dataset.

**Alur:** setup → download dataset → split train/valid/test bila perlu → train dengan augmentasi moderat → validasi → export ONNX + TensorRT fp16 → kumpulkan artefak.

Dataset lokal: `model/data/night-vision/`. Artefak dinamai `best-night-thermal.*` agar model siang tidak tertimpa.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    api_key = user_secrets.get_secret("ROBOFLOW_API_KEY")
except ImportError:
    api_key = None

DATA_DIR = Path("/kaggle/working/thermal-human-detection-uav-night")
INPUT_ZIP = next(Path("/kaggle/input").glob("*/thermal-human-detection-from-uav*.zip"), None)
print("API key tersedia:", bool(api_key))


In [ ]:
!pip install -q --upgrade ultralytics roboflow
import ultralytics, roboflow
print("ultralytics", ultralytics.__version__)

In [ ]:
if not (DATA_DIR / "data.yaml").exists():
    if api_key:
        from roboflow import Roboflow
        rf = Roboflow(api_key=api_key)
        project = rf.workspace("thermal-disasters-project").project("thermal-human-detection-from-uav")
        project.version(1).download("yolov11", location=str(DATA_DIR), overwrite=True)
    elif INPUT_ZIP:
        import zipfile
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(INPUT_ZIP) as z:
            z.extractall(DATA_DIR)
    else:
        raise SystemExit("Tidak ada API key maupun zip — upload dataset dulu (lihat markdown atas)")
print("dataset siap:", (DATA_DIR / "data.yaml").exists())


In [ ]:
import random
import shutil
import yaml

def ensure_splits():
    if all((DATA_DIR / split / "images").exists() for split in ("valid", "test")):
        return
    train_images = sorted((DATA_DIR / "train" / "images").glob("*"))
    if len(train_images) < 10:
        raise RuntimeError("Dataset thermal terlalu kecil untuk split train/valid/test")
    rng = random.Random(26)
    rng.shuffle(train_images)
    valid_count = max(1, round(len(train_images) * 0.10))
    test_count = max(1, round(len(train_images) * 0.10))
    selections = {
        "valid": train_images[:valid_count],
        "test": train_images[valid_count:valid_count + test_count],
    }
    labels = DATA_DIR / "train" / "labels"
    for split, images in selections.items():
        image_dir = DATA_DIR / split / "images"
        label_dir = DATA_DIR / split / "labels"
        image_dir.mkdir(parents=True, exist_ok=True)
        label_dir.mkdir(parents=True, exist_ok=True)
        for image in images:
            shutil.move(str(image), image_dir / image.name)
            label = labels / f"{image.stem}.txt"
            if label.exists():
                shutil.move(str(label), label_dir / label.name)

ensure_splits()
cfg = yaml.safe_load((DATA_DIR / "data.yaml").read_text())
print(cfg["names"])
for split in ["train", "valid", "test"]:
    d = DATA_DIR / split / "images"
    n = len(list(d.glob("*.jpg"))) + len(list(d.glob("*.png")))
    print(split, n)


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26s.pt")
model.train(
    data=str(DATA_DIR / "data.yaml"),
    epochs=100,
    imgsz=640,
    batch=64,           # total 2 GPU (32/GPU); auto-turun bila OOM
    patience=20,
    cache=True,
    device="0,1",
    project="/kaggle/working/run26s-thermal",
    name="train",
    exist_ok=True,
    degrees=5,
    translate=0.1,
    scale=0.4,
    fliplr=0.5,
    mosaic=0.5,
)


In [ ]:
import json
m = model.val(data=str(DATA_DIR / "data.yaml"), device=0)
metrics = {
    "mAP50-95": float(m.box.map),
    "mAP50": float(m.box.map50),
    "precision": float(m.box.mp),
    "recall": float(m.box.mr),
}
print(metrics)
Path("/kaggle/working/metrics.json").write_text(json.dumps(metrics, indent=2))

In [ ]:
best = "/kaggle/working/run26s-thermal/train/weights/best.pt"
model = YOLO(best)
model.export(format="onnx", imgsz=640, dynamic=True)
try:
    model.export(format="engine", imgsz=640, half=True)
except Exception as e:
    print("engine export gagal (opsional):", e)


In [ ]:
import shutil
out = Path("/kaggle/working/artifacts")
out.mkdir(exist_ok=True)
artifact_names = {
    "best.pt": "best-night-thermal.pt",
    "best.onnx": "best-night-thermal.onnx",
    "best.engine": "best-night-thermal.engine",
}
for source_name, output_name in artifact_names.items():
    src = Path("/kaggle/working/run26s-thermal/train/weights") / source_name
    if src.exists():
        shutil.copy(src, out / output_name)
for f in out.iterdir():
    print(f.name, f.stat().st_size // 1024, "KB")


## Setelah selesai

1. **Commit & Run** agar output tersimpan.
2. Buka tab **Output** → download folder `artifacts/` → simpan ke `D:/KRTI/model/`.
3. Verifikasi model thermal tanpa menimpa model siang:

```bash
.venv\Scripts\python -c "from ultralytics import YOLO; m = YOLO('D:/KRTI/model/best-night-thermal.pt'); r = m.predict('D:/KRTI/test_frame.jpg', conf=0.25, save=True, project='D:/KRTI/out', name='verify-night-thermal', exist_ok=True); print(len(r[0].boxes), 'deteksi')"
```

Catatan: dataset hanya 654 gambar thermal dan split valid/test dibuat deterministik 10% + 10%; cek hasil pada rekaman thermal UAV nyata sebelum dipakai operasi.
